In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(color_codes=True)

df = pd.read_csv("data/AirQualityUCI.csv", sep=';', decimal=',')
df.head(5)


**Data cleaning**


In [ ]:
df = df.drop(['Date','Time','PT08.S1(CO)','PT08.S2(NMHC)','PT08.S3(NOx)','PT08.S4(NO2)','PT08.S5(O3)'], axis=1)
df

In [ ]:
df = df.dropna(axis=1, how='all')
df

In [ ]:
df.shape
duplicate = df[df.duplicated()]
df = df.drop_duplicates()
df = df.replace(-200, np.nan)

In [ ]:
print(df.isnull().sum())
df = df.dropna()
df.count()

**Data visualization**


In [ ]:
plt.figure(figsize=(10,5))
c= df.corr()
sns.heatmap(c,cmap="BrBG",annot=True)
c

In [ ]:
sns.pairplot(df, hue='AH', height=2.5)

**Data normalization**

In [ ]:
X = df.drop(['C6H6(GT)', 'AH'], axis=1).values
y = df["C6H6(GT)"].values

In [ ]:
def scale(x):
    x = x.astype(float)
    x_scaled = x - np.mean(x, axis=0)
    x_scaled = x_scaled / np.std(x_scaled, axis=0)
    return x_scaled

In [ ]:
x_sd=scale(X)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x_sd, y, train_size=.7, random_state=1)


**Model**

In [ ]:


class LinearRegression:

    def __init__(self, l_rate=0.002, iterations=1000):
        self.l_rate = l_rate
        self.iterations = iterations

    def fitGD(self, x, y):
        self.cost = []
        self.theta = np.zeros((1 + x.shape[1]))
        n = x.shape[0]

        for i in range(self.iterations):
            y_pred = self.theta[0] + np.dot(x, self.theta[1:])
            mse = (1/n) * np.sum((y_pred - y)**2)
            self.cost.append(mse)


            d_theta1 = (2/n) * np.dot(x.T, (y_pred - y))
            d_theta0 = (2/n) * np.sum(y_pred - y)


            self.theta[1:] = self.theta[1:] - self.l_rate * d_theta1
            self.theta[0] = self.theta[0] - self.l_rate * d_theta0
        return self


    def predictGD(self, x):
        return self.theta[0] + np.dot(x, self.theta[1:])


    def fitNQ(self,x, y):
        z = np.ones((x.shape[0],1))
        x = np.append(z, x, axis=1)
        self.thetas = np.linalg.pinv(x.T.dot(x)).dot(x.T).dot(y)

    def predictNQ(self, x):
        z = np.ones((x.shape[0],1))
        x = np.append(z, x, axis=1)
        return np.dot(x, self.thetas)


In [ ]:

lr=LinearRegression()
lr.fitGD(X_train, y_train)
lr.fitNQ(X_train,y_train)

In [ ]:
print("theta_0= ", lr.theta[0])
print("theta_1= ", lr.theta[1])
print("theta_2= ", lr.theta[2])

In [ ]:

print("theta_0= ", lr.thetas[0])
print("theta_1= ", lr.thetas[1])
print("theta_2= ", lr.thetas[2])

In [ ]:
y_pred = lr.predictGD(X_test)
y_predNQ = lr.predictNQ(X_test)

In [ ]:
y_predNQ = lr.predictNQ(X_test)

**Model evaluation using R²**

In [ ]:
errors = np.sum((y_predNQ - y_test)**2)
sst = np.sum((y_test- np.mean(y_test))**2)
r2_NQ = 1 - (errors/sst)

adusted_r2_NQ= 1-((1-r2_NQ) * (X_test.shape[0] -1)/(X_test.shape[0] - X_test.shape[1] - 1))
print (adusted_r2_NQ)

In [ ]:
errors = np.sum((y_pred - y_test)**2)
sst = np.sum((y_test - np.mean(y_test))**2)
r2_GD = 1 - (errors/sst)

adusted_r2_GD= 1-((1-r2_GD) * (X_test.shape[0] -1)/(X_test.shape[0] - X_test.shape[1] - 1))
print (adusted_r2_GD)

**Loss function visualization**

In [ ]:
plt.title('loss values')
plt.plot(lr.cost)
plt.ylabel('loss')
plt.xlabel('iteration')